# Mouse EGFR top5 structural check
Private free-compute check. Original sequences, mouse Q01279 residues25-647, no MSA. One sample per sequence. No pH, affinity, glycan or binding validation. No submission.

In [ ]:



from pathlib import Path
import sys, subprocess, json, os, time, hashlib, csv, io, zipfile, math, shutil
ROOT = Path('/kaggle/working/Proteinbase_Mouse5')
ROOT.mkdir(parents=True, exist_ok=True)
JOB_START=time.time()
CACHE = Path('/kaggle/temp/Proteinbase_Mouse5_cache')
CACHE.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'Activa GPU en Settings antes de continuar.'
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory/2**30,2), 'GiB VRAM')
print('RAM host disponible:')
print(Path('/proc/meminfo').read_text().splitlines()[:3])
print('Espacio libre working GiB:',round(shutil.disk_usage(ROOT).free/2**30,2))
assert shutil.disk_usage(ROOT).free > 7*2**30, 'Hace falta espacio para instalación/outputs.'






ENV = Path('/kaggle/temp/Proteinbase_Mouse5_py312')
UVROOT = Path('/kaggle/temp/Proteinbase_uv')
subprocess.run([sys.executable,'-m','pip','install','--target',str(UVROOT),'uv==0.8.22'],check=True)
UV = str(UVROOT/'bin/uv')
RUN_ENV = dict(os.environ)
RUN_ENV['UV_PYTHON_INSTALL_DIR']='/kaggle/temp/Proteinbase_python'
RUN_ENV['UV_CACHE_DIR']='/kaggle/temp/Proteinbase_uv_cache'
subprocess.run([UV,'venv','--python','3.12',str(ENV)],env=RUN_ENV,check=True)
PYTHON = str(ENV/'bin/python')
with open(ROOT/'install.log','w') as log:
    result=subprocess.run([UV,'pip','install','--python',PYTHON,'torch==2.6.0','--index-url','https://download.pytorch.org/whl/cu124'],env=RUN_ENV,stdout=log,stderr=subprocess.STDOUT)
    if result.returncode==0:
        result=subprocess.run([UV,'pip','install','--python',PYTHON,'boltz==2.2.1','torch==2.6.0'],env=RUN_ENV,stdout=log,stderr=subprocess.STDOUT)
if result.returncode:
    print((ROOT/'install.log').read_text()[-6000:])
    raise RuntimeError('Instalación fallida. Conserva install.log.')
with open(ROOT/'versions.txt','w') as log:
    subprocess.run([UV,'pip','freeze','--python',PYTHON],env=RUN_ENV,stdout=log,check=True)
subprocess.run([str(ENV/'bin/boltz'),'predict','--help'],check=True,stdout=subprocess.DEVNULL)
print('Boltz instalado con Python3.12; versiones guardadas.')





CSV_TEXT = 'name,sequence,molecule_class\r\nr03_l07,SVTVDTYADPGDAPMVDHMAVTLGDHVEEAEVFTTENADITVRYKDKHGKVHEHTFTAPPEGTEYTVEIP,protein\r\nn03_rank03_native,SVTVTFEALPGDPPIVLSVEVTLGDHVEEAEVIADPDGTITVRYKDKHGKVHEHTFTAPPEGGVYTVEIP,protein\r\nr02_l07,DLYEQAEALAKEGKVEEALELLRKLIEEHPADLFHAILDLTIEPNGKKILEELAKEHPELKELIDEALLLN,protein\r\nr01_l05,MCDSETLADMPANSFHVFECPHGSKVIVECLEEGPDSKTLKITFINEHGEVKETTLTVKKDEALTFS,protein\r\nr02_l03,MLAEQAVLLAKEGKVEEALELLRKLIEEHPNLLPHALLDLTIFPNGKKILEELAKEHPELKELIDEALLLN,protein\r\n'
MOUSE_EGFR = 'LEEKKVCQGTSNRLTQLGTFEDHFLSLQRMYNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNALYENTYALAILSNYGTNRTGLRELPMRNLQEILIGAVRFSNNPILCNMDTIQWRDIVQNVFMSNMSMDLQSHPSSCPKCDPSCPNGSCWGGGEENCQKLTKIICAQQCSHRCRGRSPSDCCHNQCAAGCTGPRESDCLVCQKFQDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGPDYYEVEEDGIRKCKKCDGPCRKVCNGIGIGEFKDTLSINATNIKHFKYCTAISGDLHILPVAFKGDSFTRTPPLDPRELEILKTVKEITGFLLIQAWPDNWTDLHAFENLEIIRGRTKQHGQFSLAVVGLNITSLGLRSLKEISDGDVIISGNRNLCYANTINWKKLFGTPNQKTKIMNNRAEKDCKAVNHVCNPLCSSEGCWGPEPRDCVSCQNVSRGRECVEKCNILEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGIMGENNTLVWKYADANNVCHLCHANCTYGCAGPGLQGCEVWPSGPKIPS'
rows=list(csv.DictReader(io.StringIO(CSV_TEXT)))
assert len(rows)==5 and len({r['name'] for r in rows})==5 and len({r['sequence'] for r in rows})==5
assert len(MOUSE_EGFR)==623
assert all(set(r['sequence'])<=set('ACDEFGHIKLMNPQRSTVWY') and r['molecule_class']=='protein' for r in rows)
INPUTS=ROOT/'inputs';INPUTS.mkdir(exist_ok=True)
(ROOT/'pepe_sequences.csv').write_text(CSV_TEXT)
for r in rows:
    (INPUTS/(r['name']+'.yaml')).write_text('version: 1\nsequences:\n  - protein:\n      id: A\n      sequence: '+r['sequence']+'\n      msa: empty\n  - protein:\n      id: B\n      sequence: '+MOUSE_EGFR+'\n      msa: empty\n')
provenance={'boltz':'2.2.1','target':'mouse EGFR ECD, 623 aa, UniProt Q01279 residues25-647','target_source':'https://rest.uniprot.org/uniprotkb/Q01279.fasta','input_sha256':hashlib.sha256(CSV_TEXT.encode()).hexdigest(),'msa':'empty, no remote MSA','samples':1,'recycling_steps':3,'sampling_steps':200,'seed':20261006,'pH_prediction':False,'affinity_prediction':False,'mouse_prediction':True,'sequence_changes':False}
(ROOT/'provenance.json').write_text(json.dumps(provenance,indent=2))
print('20 YAML preparados. Primera secuencia:',rows[0]['name'])






WRAPPER = "import os,sys,time,threading,json,hashlib,collections,typing\nfrom pathlib import Path\nimport torch\nfrom omegaconf.dictconfig import DictConfig\nfrom omegaconf.listconfig import ListConfig\nfrom omegaconf.base import ContainerMetadata,Metadata\nfrom omegaconf.nodes import AnyNode\ncache=Path(os.environ['BOLTZ_CACHE'])\nroot=Path(os.environ['PB_ROOT'])\nfrom boltz.main import download_boltz2\n# Descarga estándar del paquete. No se envían secuencias.\ndownload_boltz2(cache)\np=cache/'boltz2_conf.ckpt'\nallowed=[DictConfig,ListConfig,ContainerMetadata,Metadata,AnyNode,int,list,dict,typing.Any,collections.defaultdict]\nif hasattr(torch.serialization,'get_unsafe_globals_in_checkpoint'):\n    observed=set(torch.serialization.get_unsafe_globals_in_checkpoint(p))\n    names={'omegaconf.dictconfig.DictConfig','omegaconf.listconfig.ListConfig','omegaconf.base.ContainerMetadata','omegaconf.base.Metadata','omegaconf.nodes.AnyNode','builtins.int','builtins.list','builtins.dict','typing.Any','collections.defaultdict'}\n    assert observed<=names, 'Clases no revisadas en checkpoint: '+str(observed-names)\ntorch.serialization.add_safe_globals(allowed)\nwith p.open('rb') as f:\n    h=hashlib.file_digest(f,'sha256').hexdigest()\n(root/'checkpoint_sha256.txt').write_text(h)\nmemtotal=int(next(l.split()[1] for l in open('/proc/meminfo') if l.startswith('MemTotal:')))\ncap=min(22*1024**2,int(memtotal*.85))\ndef guard():\n    while True:\n        with open('/proc/self/status') as f:\n            rss=int(next(l.split()[1] for l in f if l.startswith('VmRSS:')))\n        if rss>cap:\n            print('RAM_GUARD_STOPPED',rss,'KiB',flush=True);os._exit(3)\n        time.sleep(.5)\nthreading.Thread(target=guard,daemon=True).start()\nprint('SAFE_ALLOWLIST; RAM_CAP_KiB',cap,flush=True)\nfrom boltz.main import cli\ncli()\n"
(ROOT/'safe_predict.py').write_text(WRAPPER)
RUN_ENV=os.environ.copy()
RUN_ENV.update({'CUDA_VISIBLE_DEVICES':'0','OMP_NUM_THREADS':'4','BOLTZ_CACHE':str(CACHE),'PB_ROOT':str(ROOT),'TORCH_FORCE_WEIGHTS_ONLY_LOAD':'1'})
# Explicitly do not disable the safe weights-only loader.
assert RUN_ENV.get('TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD') not in ('1','true','True'), 'Quita ese override inseguro.'



RECORDS=ROOT/'run_records.json'
records=json.loads(RECORDS.read_text()) if RECORDS.exists() else []
def archive():
    files=[p for p in ROOT.rglob('*') if p.is_file()]
    (ROOT/'SHA256SUMS.txt').write_text('\n'.join(hashlib.sha256(p.read_bytes()).hexdigest()+'  '+str(p.relative_to(ROOT)) for p in files if p.name!='SHA256SUMS.txt'))
    out=Path('/kaggle/working/Proteinbase_Mouse5_results.zip')
    with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
        for p in ROOT.rglob('*'):
            if p.is_file():z.write(p,str(p.relative_to(ROOT)))
    print('Respaldo:',out)
def run_one(r):
    assert time.time()-JOB_START<85*60, 'Time guard: not starting another fold near quota limit'
    name=r['name'];out=ROOT/'predictions'/name
    existing=list(out.rglob('confidence_*_model_*.json')) if out.exists() else []
    structures=list(out.rglob('*_model_*.cif')) if out.exists() else []
    if existing and structures:
        print('Ya tiene salida, no se repite:',name);return None
    logpath=ROOT/(name+'.log');start=time.time()
    cmd=[PYTHON,str(ROOT/'safe_predict.py'),'predict',str(INPUTS/(name+'.yaml')),'--out_dir',str(out),'--cache',str(CACHE),'--model','boltz2','--diffusion_samples','1','--max_parallel_samples','1','--num_workers','0','--seed','20261006','--recycling_steps','3','--sampling_steps','200','--write_full_pae']
    try:
        with open(logpath,'w') as log:
            result=subprocess.run(cmd,env=RUN_ENV,stdout=log,stderr=subprocess.STDOUT,timeout=2400)
        elapsed=time.time()-start
        conf=list(out.rglob('confidence_*_model_*.json'));structures=list(out.rglob('*_model_*.cif'))
        ok=result.returncode==0 and bool(conf) and bool(structures)
        rec={'name':name,'seconds':elapsed,'returncode':result.returncode,'completed':ok}
        if ok:
            data=json.loads(conf[0].read_text())
            rec['confidence']=data
            print(name,'terminado',round(elapsed/60,1),'min; ipTM',data.get('iptm'))
        else:print(logpath.read_text()[-5000:])
        records.append(rec);RECORDS.write_text(json.dumps(records,indent=2))
        if not ok:raise RuntimeError(name+': no hay estructura y confianza completas; no continuar a ciegas.')
        return elapsed
    except subprocess.TimeoutExpired:
        records.append({'name':name,'completed':False,'failure':'timeout 40 min'});RECORDS.write_text(json.dumps(records,indent=2))
        raise
    finally:archive()






pilot_seconds=run_one(rows[0])
if pilot_seconds is not None:
    print('Escenario conservador para las otras19:',round(19*pilot_seconds/3600,2),'horas, usando el tiempo del piloto incluida descarga.')






assert list((ROOT/'predictions'/rows[0]['name']).rglob('confidence_*_model_*.json')), 'Piloto sin confianza.'
assert list((ROOT/'predictions'/rows[0]['name']).rglob('*_model_*.cif')), 'Piloto sin estructura.'
for r in rows[1:]:
    run_one(r)
archive()
print('Terminado. Esto NO prueba afinidad, pH-selectividad ni unión a ratón.')




